# 🧬 BioKG Tutorial Part 1: Exploratory Data Analysis & Scale

Welcome to the BioKG Link Predictor series! This is a masterclass in building a **Production-Grade Deep Learning system on Biological Data**. 

### 🎯 Our Goal:
Can we predict un-discovered biological relationships? If a drug interacts with a specific protein, and that protein controls a disease, we might discover a brand new **Drug Repurposing** link. This exact methodology was used during the COVID-19 pandemic to find existing antiviral candidates in days instead of years.

### 📊 The Dataset: PrimeKG (Precision Medicine Knowledge Graph)
We are using Harvard's **PrimeKG**, a massive integration of 20 distinct biomedical databases. It contains:
- ~129,000 biological nodes (Proteins, Diseases, Drugs, Pathways)
- ~8.1 million relationships (edges) connecting them

---
### ⚡ Step 0: Ensure GPU is Active
Before we run step-by-step, make sure your Colab environment is utilizing an NVIDIA T4 GPU.
*(Runtime -> Change Runtime Type -> T4 GPU)*

In [ ]:
import subprocess

gpu_info = subprocess.getoutput('nvidia-smi')
if 'NVIDIA-SMI' not in gpu_info:
    print('❌ Warning: You are not connected to a GPU.')
else:
    print('✅ GPU detected! You are connected to:\n')
    print('\n'.join(gpu_info.split('\n')[7:9]))  # Print just the GPU name

---
## 📥 Step 1: Downloading the Raw Knowledge Graph
Let's fetch the actual file from the Harvard Dataverse. PrimeKG ships the entire network of 8.1M edges neatly inside a massive uncompressed `kg.csv` file.

In [ ]:
import os
import requests
from pathlib import Path
from tqdm.auto import tqdm

DATA_DIR = Path('data/raw/primekg')
DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_file(url, dest_path, desc):
    if dest_path.exists():
        print(f"⏭️ {dest_path.name} already exists.")
        return
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(desc=desc, total=total, unit='iB', unit_scale=True) as pb:
        for chunk in response.iter_content(8192):
            pb.update(f.write(chunk))
    print(f"✅ Saved: {dest_path.name}")

print("Fetching PrimeKG...")
download_file("https://dataverse.harvard.edu/api/access/datafile/6180620", DATA_DIR / 'kg.csv', 'PrimeKG Graph')

---
## 🔍 Step 2: Exploratory Data Analysis (EDA) on Nodes
A Knowledge graph is made of **Nodes** (the entities) and **Edges** (the links). 
Let's figure out exactly what kind of biological data we are working with by extracting the Unique Nodes out of the edge connections.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

print("Loading the PrimeKG Graph for EDA...")
df_kg = pd.read_csv(DATA_DIR / 'kg.csv', low_memory=False)

# Extract unique nodes by combining the source (X) and target (Y) entities from the edge list
x_nodes = df_kg[['x_index', 'x_type', 'x_name']].rename(columns={'x_index':'node_index', 'x_type':'node_type', 'x_name':'node_name'})
y_nodes = df_kg[['y_index', 'y_type', 'y_name']].rename(columns={'y_index':'node_index', 'y_type':'node_type', 'y_name':'node_name'})
df_nodes = pd.concat([x_nodes, y_nodes]).drop_duplicates(subset=['node_index'])

print(f"Total Unique Biological Entities: {len(df_nodes):,}\n")
display(df_nodes.head(3))

### 📊 Plotting Entity Distribution
What types of biology are most represented here?

In [ ]:
plt.figure(figsize=(10, 6))
type_counts = df_nodes['node_type'].value_counts()

ax = sns.barplot(x=type_counts.values, y=type_counts.index, palette="viridis")
plt.title('Distribution of Biological Entity Types in PrimeKG', fontsize=16, pad=15)
plt.xlabel('Number of Nodes', fontsize=12)
plt.ylabel('Entity Category', fontsize=12)

# Add physical numbers to the bars to make it easy to read
for i, v in enumerate(type_counts.values):
    ax.text(v + 1000, i + 0.1, f"{v:,}", color='black', fontweight='bold')

plt.xlim(0, max(type_counts.values) * 1.2)
plt.tight_layout()
plt.show()

# Notice that Gene/Protein represent the massive majority of nodes in biological systems!

---
## ⚡ Step 3: Tackling 8.1 Million Edges & GPU Acceleration

We have explored our 129,000 nodes. Now we need to process the 8,100,000 connections (Edges).

If we use standard Pandas (CPU), manipulating 8.1M rows to format our ML pipeline can take minutes or even crash Colab's limited RAM. 
**To solve this, we will use NVIDIA RAPIDS (cuDF)** natively within Colab to process millions of rows directly on the T4 GPU Core!


In [ ]:
import torch

print("📦 Installing RAPIDS (cuDF)... Give this ~3 minutes to compile.")

cuda_version = torch.version.cuda
major = int(cuda_version.split('.')[0]) if cuda_version else 12
rapids_suffix = 'cu11' if major == 11 else 'cu12'

!pip install cudf-{rapids_suffix} --extra-index-url=https://pypi.nvidia.com -q

print("✅ cuDF Installed!")

### 🏎️ CPU vs GPU Benchmark
Let's race loading 8.1 million rows. `pandas` vs `cuDF`.

In [ ]:
import time
import cudf

edges_file = str(DATA_DIR / 'kg.csv')
print('⚡ Starting PrimeKG Edges Benchmark...')

# 🐢 Pandas CPU Benchmark
t0 = time.time()
pdf = pd.read_csv(edges_file, low_memory=False)
cpu_time = time.time() - t0
print(f'🐢 pandas (CPU): {cpu_time:.2f} seconds | {len(pdf):,} edges loaded in RAM')

# 🚀 cuDF GPU Benchmark
t0 = time.time()
gdf = cudf.read_csv(edges_file)
gpu_time = time.time() - t0
print(f'⚡ cuDF (GPU):   {gpu_time:.2f} seconds | {len(gdf):,} edges loaded in VRAM')

speedup = cpu_time / max(gpu_time, 0.001)
print(f'\n🚀 Speedup Factor: {speedup:.1f}x Faster Data Engineering \n')
display(gdf.head(3))

---
## 💾 Step 4: Structuring for MLOps (Parquet)

PyTorch expects arrays of Numbers, not Strings. 
We need to extract the `head_index` (the integer ID of the source node), the `tail_index`, and map the string `relation` to an integer.

We then save it to a binary **Parquet** format. Parquet is the industry standard for saving huge data frames—it compresses heavily and preserves datatypes.

In [ ]:
import json

# 1. Create mapping from Relation Strings to Integer IDs
# (Doing this on Pandas data representation for simplicity here)
unique_relations = pdf['relation'].unique()
relation2id = {str(rel): int(idx) for idx, rel in enumerate(sorted(unique_relations))}

print(f"Found {len(relation2id)} unique relationship edges in PrimeKG.")

# 2. Map relation string to relation_id using cuDF for lightning speed
# First covert python dict to cuDF dataframe to merge
mapping_df = cudf.DataFrame({'relation': list(relation2id.keys()), 'relation_id': list(relation2id.values())})
clean_gdf = gdf.merge(mapping_df, on='relation', how='left')

# 3. Save as Parquet & JSON metadata for the Training Modules
PROC_DIR = Path('data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

# PrimeKG natively uses 'x_index' and 'y_index'. We align these to standard graph nomenclature
final_df = clean_gdf[['x_index', 'relation_id', 'y_index']].rename(columns={'x_index': 'head_index', 'y_index': 'tail_index'})
final_df.to_parquet(PROC_DIR / 'primekg_edges.parquet')

with open(PROC_DIR / 'relation2id.json', 'w') as f:
    json.dump(relation2id, f, indent=2)

stats = {
    "max_node_index": int(df_nodes['node_index'].max()),
    "n_relations": len(relation2id)
}
with open(PROC_DIR / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)

print("\n✅ ML-ready Graph Data Processed and Saved as Parquet!")

### 🎉 Summary of Part 1
- We explored biological nodes, revealing that the system is centered mostly on interacting Genes and Proteins.
- We proved how transferring string manipulation workloads from Pandas to NVIDIA cuDF accelerates big-data operations drastically.
- We set up our integer-based arrays, ready for PyTorch modeling.

➡ **Proceed to Tutorial 2**: We will introduce `BioBridge` to map meaning onto these raw integers using multimodal Large Language Models (LLMs).